In [ ]:
from pathlib import Path
import os
import json
import time
import random
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

RANDOM_STATE = 11126
TRAIN_VAL_PATH = 'Swissprot_Train_Validation_dataset.csv'
HPA_TEST_PATH = 'hpa_testset.csv'

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

from train_esm import (
    load_splits,
    ProteinDataset,
    collect_metrics,
    run_epoch,
    set_seed,
    ProteinFoundationClassifier,
    configure_trainability,
    masked_mean_pool,
)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

PyTorch: 2.11.0
Device: mps


# Homework — sequence models and ESM-2

## Objective

Investigate the trade-off between model size, amount of fine-tuning, predictive performance and computational cost for protein subcellular-localization prediction. The task remains **multi-label classification** with the eight localization labels used in this practical.

> **Evaluation rule:** use the SwissProt training/validation partitions for model development. Keep `hpa_testset.csv` untouched until the final evaluation. Do not tune hyperparameters or thresholds on HPA.

## Part A — Reproduce and document a baseline

1. Re-run either the LSTM or BiLSTM baseline.
2. Re-run the local ESM-2 8M frozen-embedding logistic probe.
3. Record the random seed, maximum sequence length, batch size, number of epochs and decision threshold.
4. Report validation and HPA test **macro-F1**, **micro-F1** and exact-match accuracy.

Answer:

- Which metric is most informative for this imbalanced multi-label problem, and why?
- Which localization labels are easiest and hardest for the baseline?
- Does the model tend to miss labels or produce false positive labels?

## Part B — Data-efficiency experiment

Train the same downstream model using 10%, 25%, 50% and 100% of the SwissProt training data. Use the same validation set, test set, architecture, preprocessing and random seed policy for every fraction.

Create a plot of validation macro-F1 against the number of training proteins. Add the HPA test score for the final selected model only.

Answer:

- At which training fraction do the performance gains begin to level off?
- Does the data-efficient model generalize to HPA in the same way as it performs on validation?
- What does this suggest about the value of additional labeled proteins?

## Part C — Frozen versus fine-tuned ESM-2

On the GPU server, compare the following experiments with the same data split and evaluation procedure:

1. Frozen ESM-2 35M with a trainable classification head.
2. Partial fine-tuning: unfreeze the last two Transformer blocks.
3. Full fine-tuning of ESM-2 35M.

Start with a one-epoch smoke test. Run the three experiments sequentially, in separate `tmux` sessions or with separate output directories. Use a smaller batch size or maximum sequence length if GPU memory is insufficient.

For each run, record:

- trainable parameter count;
- validation and HPA test macro-F1;
- micro-F1 and exact-match accuracy;
- training time per epoch;
- peak GPU memory;
- the best validation epoch;
- whether training and validation curves indicate overfitting.

Use this table in the report:

| Model | Training mode | Trainable parameters | Max length | Batch size | Val macro-F1 | HPA macro-F1 | Time/epoch | Peak GPU memory |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| LSTM or BiLSTM | from scratch | | | | | | | |
| ESM-2 8M + logistic probe | frozen embeddings | | | | | | | |
| ESM-2 35M + head | frozen | | | | | | | |
| ESM-2 35M | partial | | | | | | | |
| ESM-2 35M | full | | | | | | | |

Answer:

- Does fine-tuning improve over a frozen backbone?
- Is the improvement large enough to justify the additional compute?
- Which model has the best validation-to-HPA generalization?
- Which classes benefit from fine-tuning, and which remain difficult?
- What evidence of overfitting can you see in the learning curves?

## Part D — Threshold and error analysis

For the best model, evaluate thresholds of 0.20, 0.35, 0.50 and 0.65. Select a threshold using the validation set only, then report the selected threshold once on HPA.

Include:

- macro-F1 as a function of threshold;
- one confusion matrix per localization label;
- a table of at least ten proteins with their true and predicted labels;
- two examples of false positive or false negative predictions and a biological interpretation.

Answer:

- Is 0.5 a reasonable threshold for every localization?
- Would separate thresholds for different labels be justified?
- Which errors could be caused by class imbalance rather than by the sequence representation?

## Part E — Split-sensitivity experiment

Create one additional random train/validation split from the SwissProt data. Check and report the prevalence of every label in both splits. Keep HPA unchanged.

Compare the validation score from the provided partition with the score from the random split. Do not describe the supplied partition as homology-aware unless this is verified from the metadata; instead discuss why random protein splits can be optimistic when related proteins occur in both sets.

Answer:

- How much does the validation score change?
- Which label prevalences change between the two splits?
- Why might a random protein split overestimate real-world generalization?

## Optional advanced extension — PEFT / LoRA

Implement LoRA or another parameter-efficient fine-tuning method for ESM-2 35M. Compare it with full fine-tuning using the same maximum sequence length and number of epochs. Report trainable parameters, GPU memory, time per epoch and macro-F1.

ESM-2 150M is optional and should be attempted only as a frozen-embedding or PEFT experiment. It is not a required full-fine-tuning task on a 12 GB GPU.

## Submission

Submit:

2. a report containing the results table, learning curves, t-SNE or embedding plot, confusion matrices and answers to the questions;


## Split

In [ ]:
train_df, val_df, hpa_df = load_splits(Path(TRAIN_VAL_PATH), Path(HPA_TEST_PATH))